# GPU Architecture Analysis
**Research question:** Do newer desktop GPUs achieve higher theoretical FP32 performance at comparable listed TDP?

**H₀:** β_year = 0 after accounting for TDP. **H₁:** β_year > 0. Significance level: 5%.

Scope: consumer desktop reference entries, 2016–2025. FP32 is theoretical peak throughput, not measured application performance; listed TDP/TGP is a power rating, not measured consumption.

**Source:** [Intelion / GPU Ark specifications dataset](https://huggingface.co/datasets/Intelion/gpuark-gpu-dataset), GPU Ark (2026), CC BY 4.0.

## 1. Data preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from IPython.display import display

df = pd.read_csv("hf://datasets/Intelion/gpuark-gpu-dataset/gpuark-gpu-specs.csv")
print("Rows and columns:", df.shape)

columns = ["id", "name", "slug", "vendor", "arch_name", "is_retail_board",
           "bus_interface", "slot_width", "card_release_date",
           "tdp", "fp32_performance", "proc_transistors"]

display(pd.DataFrame({"type": df[columns].dtypes, "missing": df[columns].isna().sum()}))
print("Exact duplicates:", df.duplicated().sum())
print("Repeated IDs:", df["id"].duplicated().sum())
df[["name", "card_release_date", "tdp", "fp32_performance", "proc_transistors"]].head()

| Source / analysis variable | Meaning | Unit | Stevens scale |
|---|---|---|---|
| id, name, slug | Identifier, product name and source-page slug | None | Nominal |
| vendor, arch_name | Chip vendor and architecture | Categories | Nominal |
| is_retail_board | Partner/OEM edition flag | Boolean | Nominal |
| bus_interface, slot_width | Interface and form factor; classification evidence | Categories | Nominal |
| card_release_date → year | Listed release date and calendar year | Date / year | Interval |
| tdp → tdp_w | Listed TDP/TGP rating | W | Ratio |
| fp32_performance → fp32_tflops | Theoretical peak FP32 throughput | TFLOPS | Ratio |
| proc_transistors → transistors_billion | Transistor count | Millions → billions | Ratio |
| group; period (derived) | Product group; ordered release-period bands | Categories | Nominal; ordinal |

Exact duplicates and partner/OEM retail-board entries are excluded. Unreadable, non-positive or infinite specifications become missing; release years are restricted to 1998–2025. Complete TDP, FP32 and year are required for both main models. Transistor missingness affects only the secondary analysis.

In [ ]:
unique = df.drop_duplicates()
assert not unique["id"].duplicated().any(), "Repeated IDs need inspection."
gpu = unique.loc[unique["is_retail_board"] == False, columns].copy()

gpu["release_date"] = pd.to_datetime(gpu["card_release_date"], errors="coerce")
gpu["year"] = gpu["release_date"].dt.year
bad_year = gpu["year"].notna() & ~gpu["year"].between(1998, 2025)
gpu.loc[bad_year, "year"] = np.nan

gpu["tdp_w"] = pd.to_numeric(gpu["tdp"].str.replace(" W", "", regex=False), errors="coerce")
gpu["fp32_tflops"] = pd.to_numeric(gpu["fp32_performance"], errors="coerce")
gpu["transistors_billion"] = pd.to_numeric(
    gpu["proc_transistors"].str.replace(",", "", regex=False)
                           .str.replace(" million", "", regex=False), errors="coerce"
) / 1000

numeric = ["tdp_w", "fp32_tflops", "transistors_billion"]
invalid = (gpu[numeric] <= 0) | gpu[numeric].isin([np.inf, -np.inf])
gpu[numeric] = gpu[numeric].mask(invalid)
print("Invalid release years:", bad_year.sum())
print("Invalid numeric values:")
print(invalid.sum())
print("Missing/unreadable after conversion:")
print(gpu[["year"] + numeric].isna().sum())

## 2. GPU classification and sample
Desktop: consumer family plus physical PCIe card. Laptop: mobile/MXM evidence. Data-center: server naming or SXM/OAM form factor. Conflicting evidence and unmatched products remain **Unknown**. Group summaries cover 1998–2025; the desktop analysis covers 2016–2025.

In [ ]:
name = gpu["name"].fillna("")
slot = gpu["slot_width"].fillna("")
bus = gpu["bus_interface"].fillna("")

physical_card = slot.isin(["Single-slot", "Dual-slot", "Triple-slot", "Quad-slot"])
consumer = name.str.contains(r"GeForce\b|Radeon (?:RX|HD|R[579])\b|Intel Arc [AB]\d{3}\b", case=False)
mobile_name = name.str.contains(r"Laptop|Mobile|Mobility", case=False)
mobile_model = name.str.contains(r"\dM\b|\bM\d{3,4}(?:X)?\b", case=False)
mobile = mobile_name | (mobile_model & (slot.eq("MXM Module") | bus.str.contains("MXM")))
server = name.str.contains(r"Data Center|Instinct|Tesla|Server|Subsystem|\bSXM|\bOAM", case=False)
server = server | slot.isin(["SXM Module", "OAM Module"])

gpu["group"] = "Unknown"
gpu.loc[mobile & ~server & ~physical_card, "group"] = "Laptop"
gpu.loc[server & ~mobile, "group"] = "Data-center / large-scale"
gpu.loc[consumer & physical_card & bus.str.startswith("PCIe") & ~mobile & ~server, "group"] = "Desktop"
gpu.loc[name.str.contains(r"Mac Edition|\bMac Pro\b", case=False), "group"] = "Unknown"

multi = name.str.contains(r"(?:GX2|X2)\b|\b(?:Dual|Duo)\b", case=False)
gpu.loc[multi & (gpu["group"] == "Desktop"), "group"] = "Unknown"

groups = ["Desktop", "Laptop", "Data-center / large-scale", "Unknown"]

# Group summaries use complete year, TDP and FP32 values across 1998–2025.
complete = gpu.dropna(subset=["year", "tdp_w", "fp32_tflops"])
group_stats = complete.groupby("group").agg(
    complete_rows=("id", "size"),
    mean_FP32=("fp32_tflops", "mean"),
    median_FP32=("fp32_tflops", "median"),
    std_FP32=("fp32_tflops", "std"),
    median_TDP=("tdp_w", "median"),
    max_TDP=("tdp_w", "max")
).reindex(groups)
group_stats.insert(0, "all_rows", gpu["group"].value_counts().reindex(groups))
group_stats.round(2)

In [ ]:
desktop_candidates = gpu[(gpu["group"] == "Desktop") & gpu["year"].between(2016, 2025)]
desktop = desktop_candidates.dropna(subset=["tdp_w", "fp32_tflops"]).copy()
desktop["year"] = desktop["year"].astype(int)
desktop["year_since_2016"] = desktop["year"] - 2016
desktop["period"] = pd.cut(desktop["year"], [2015, 2018, 2021, 2025],
                          labels=["2016–2018", "2019–2021", "2022–2025"])

row_counts = pd.Series({
    "Raw rows": len(df),
    "After exact duplicate removal": len(unique),
    "Reference entries": len(gpu),
    "Classified desktop entries (all dates)": (gpu["group"] == "Desktop").sum(),
    "Desktop entries dated 2016–2025": len(desktop_candidates),
    "Desktop rows complete for BOTH models": len(desktop)
}, name="Rows")
display(row_counts.to_frame())
print("Missing FP32 or TDP excluded:", len(desktop_candidates) - len(desktop))
print("Distinct years:", desktop["year"].nunique())

# Inspect extreme ratings without deleting valid high-end products.
print("Largest catalog TDP:", gpu.loc[gpu["tdp_w"].idxmax(), ["name", "group", "tdp_w"]].to_dict())
variables = ["tdp_w", "fp32_tflops", "transistors_billion"]
q1, q3 = desktop[variables].quantile(0.25), desktop[variables].quantile(0.75)
iqr = q3 - q1
outliers = (desktop[variables] < q1 - 1.5 * iqr) | (desktop[variables] > q3 + 1.5 * iqr)
print("IQR flags (retained):", outliers.sum().to_dict())

In [ ]:
display(desktop[["year", "tdp_w", "fp32_tflops", "transistors_billion"]]
        .describe().round(3))

similar = desktop[desktop["tdp_w"].between(150, 200)].copy()
band_stats = similar.groupby("period", observed=True).agg(
    n=("id", "size"), min_TDP=("tdp_w", "min"), max_TDP=("tdp_w", "max"),
    median_TDP=("tdp_w", "median"), median_FP32=("fp32_tflops", "median")
).round(3)
display(band_stats)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
points = axes[0].scatter(desktop["year"], desktop["fp32_tflops"],
                         c=desktop["tdp_w"], cmap="viridis", s=30, alpha=0.8)
axes[0].set_xlabel("Release year")
axes[0].set_ylabel("Peak FP32 (TFLOPS)")
axes[0].set_title("Consumer desktop GPUs, 2016–2025")
fig.colorbar(points, ax=axes[0], label="Listed TDP (W)")

sns.boxplot(data=similar, x="period", y="fp32_tflops", color="#88BBDD", ax=axes[1])
axes[1].set_xlabel("Release period")
axes[1].set_ylabel("Peak FP32 (TFLOPS)")
axes[1].set_title("Desktop GPUs rated 150–200 W")
axes[2].hist(desktop["fp32_tflops"], bins=15, color="#4477AA", edgecolor="white")
axes[2].set_xlabel("Theoretical FP32 (TFLOPS)")
axes[2].set_ylabel("Number of desktop entries")
axes[2].set_title("Desktop FP32 distribution")
plt.tight_layout()
plt.show()

In [ ]:
y = desktop["fp32_tflops"]
X1 = sm.add_constant(desktop[["tdp_w"]])
X2 = sm.add_constant(desktop[["tdp_w", "year_since_2016"]])
m1 = sm.OLS(y, X1).fit()
m2 = sm.OLS(y, X2).fit()
comparison = pd.DataFrame({
    "n": [int(m1.nobs), int(m2.nobs)],
    "Intercept": [m1.params["const"], m2.params["const"]],
    "TDP (TFLOPS/W)": [m1.params["tdp_w"], m2.params["tdp_w"]],
    "Year (TFLOPS/year)": [np.nan, m2.params["year_since_2016"]],
    "R²": [m1.rsquared, m2.rsquared]
}, index=["FP32 ~ TDP", "FP32 ~ TDP + year"])
display(comparison.round(3))

robust = m2.get_robustcov_results(cov_type="HC3")
year_index = list(m2.params.index).index("year_since_2016")
year_coef = m2.params["year_since_2016"]
p_two_sided = robust.pvalues[year_index]
p_one_sided = p_two_sided / 2 if year_coef > 0 else 1 - p_two_sided / 2
ci_low, ci_high = robust.conf_int()[year_index]

print(f"Year coefficient: {year_coef:.3f} TFLOPS/year")
print(f"HC3 two-sided 95% confidence interval: [{ci_low:.3f}, {ci_high:.3f}]")
print(f"HC3 one-sided p-value for H1: {p_one_sided:.3g}")
print("Reject H0 at 5% (subject to assumptions):", p_one_sided < 0.05)
print(f"Simple model: FP32 = {m1.params['const']:.3f} + {m1.params['tdp_w']:.3f} × TDP.")
print(f"Multiple model: FP32 = {m2.params['const']:.3f} + {m2.params['tdp_w']:.3f} × TDP + {year_coef:.3f} × (year − 2016).") 

## 5. Fitted relationship and residuals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(desktop["tdp_w"], y, s=25, alpha=0.6)
order = desktop["tdp_w"].sort_values().index
axes[0].plot(desktop.loc[order, "tdp_w"], m1.fittedvalues.loc[order],
             color="#CC5533", label=f"OLS; R² = {m1.rsquared:.3f}")
axes[0].set_xlabel("Listed TDP (W)")
axes[0].set_ylabel("Theoretical FP32 (TFLOPS)")
axes[0].set_title("Simple regression: FP32 vs TDP")
axes[0].legend()
axes[1].scatter(m2.fittedvalues, m2.resid, alpha=0.6, s=25)
axes[1].axhline(0, color="#CC5533", linestyle="--")
axes[1].set_xlabel("Fitted FP32 (TFLOPS)")
axes[1].set_ylabel("Residual (TFLOPS)")
axes[1].set_title("Multiple regression: residuals")
plt.tight_layout()
plt.show()
print("Negative fitted values:", int((m2.fittedvalues < 0).sum()))

without_5090 = desktop[~desktop["name"].str.contains("RTX 5090", case=False)]
m_check = sm.OLS(without_5090["fp32_tflops"],
                 sm.add_constant(without_5090[["tdp_w", "year_since_2016"]])).fit()
print(f"Sensitivity only: omitting RTX 5090 family gives n={int(m_check.nobs)}, year slope={m_check.params['year_since_2016']:.2f}, R²={m_check.rsquared:.3f}; main sample unchanged.")

Residual curvature and unequal spread indicate an imperfect linear fit. Negative predictions are physically implausible and reflect poor fit near the low end. HC3 does not correct this structure or dependence among related variants.

## 6. Secondary result: transistor count
The transistor regression describes hardware scale and theoretical FP32; it does not isolate architecture or generation effects.

In [ ]:
transistor_data = desktop.dropna(subset=["transistors_billion"]).copy()
print("Rows in transistor analysis:", len(transistor_data))
X3 = sm.add_constant(transistor_data[["transistors_billion"]])
m3 = sm.OLS(transistor_data["fp32_tflops"], X3).fit()
print(f"Intercept = {m3.params['const']:.3f}; slope = {m3.params['transistors_billion']:.3f} TFLOPS/billion; R² = {m3.rsquared:.3f}")

## 7. Conclusion

In [ ]:
print(f"Among {len(desktop)} desktop entries, adding year increases R² from {m1.rsquared:.3f} to {m2.rsquared:.3f}.")
print(f"At fixed listed TDP, each later year is associated with {year_coef:.2f} additional TFLOPS;")
print(f"an additional 100 W is associated with {100 * m2.params['tdp_w']:.2f} TFLOPS at fixed year.")
print(f"Year test: one-sided p = {p_one_sided:.3g}; " +
      ("reject H₀ in favor of H₁ at 5%." if p_one_sided < 0.05 else "do not reject H₀ at 5%."))
print("150–200 W median FP32 (TFLOPS):", band_stats["median_FP32"].to_dict())

Newer desktop entries show higher theoretical FP32 at comparable listed TDP. This association is conditional on the catalog sample and linear-model assumptions. Related GPU variants may not be independent; missing specifications and conservative classification restrict coverage. Listed power definitions differ across vendors. The results do not establish causation, measured workload performance or measured energy efficiency.